# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys


# run pip install scikit-fingerprints --> won't work
#from skfp.fingerprints import MordredFingerprint, LayeredFingerprint

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [2]:
df = pd.read_csv("ames_data.csv")
df.head()
#df.describe()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

smile="O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2"
mol = Chem.MolFromSmiles(smile)
fp = GetRDKFingerprint(mol)
display(fp)
display(rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol))
gen=TopologicalTorsionGenerator()
display(gen.GetTTFingerprint(mol))

In [ ]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def morganfp1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

def RDK(mol):
    fpg = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=5)
    ao = rdFingerprintGenerator.AdditionalOutput()
    ao.AllocateBitPaths()
    fp = fpg.GetFingerprint(mol,additionalOutput=ao)
    #fp = fp_mordred.transform(mol)
    return np.array(fp)

#def topological_torsion(mol):
 #   fp=GetTopologicalTorsionGenerator.GetTTFingerprint(mol)
  #  return np.array(fp)


"""
def rdkitfp(mol):
    fp = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def atompairfp(mol):
    fp = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)
    
    """
fpgens = {
    "MorganFP": morganfp,
    "MACCSkeys": maccskeys,
    "RDKitFP": RDK,
    "morganfp1024": morganfp1024
   # "TopologicalTorsion": topological_torsion
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MACCSkeys,RDKitFP,morganfp1024
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x34c75a420>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x34c75a880>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x34c75a9d0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x34c75aa40>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x34c75aab0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [25]:
X = np.stack(df["RDKitFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

#display(X)
#display(y)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

>Output for binary classification Sigmoid With BCEWithLogitsLoss

In [6]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size_1 = 25
hidden_size_2 = 15
output_size = 1


In [7]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(input_size, hidden_size_1)  # input layer with width = length of the feature set
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_1)  # one hidden layer, try to add another one
        self.fc3 = nn.Linear(hidden_size_1, hidden_size_2)  # one hidden layer, try to add another one
        self.fc4 = nn.Linear(hidden_size_2, hidden_size_2)  # one hidden layer, try to add another one

        self.fco = nn.Linear(hidden_size_2, output_size)  # output layer

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))  # relu activation function for input layer
        x = torch.relu(self.fc2(x))  # relu activation function for hidden layer
        x = torch.relu(self.fc3(x))  # relu activation function for hidden layer
        x = torch.relu(self.fc4(x))  # relu activation function for hidden layer
        
        #x = torch.sigmoid(self.fco(x)) for BCELoss
        x = self.fco(x)
        return x


<blockquote> Johannes did 2028 -> 256 -> 64 -> 1
+ defined drop out after every fct

self.fc2 = nn.Linear(256, 64)
self.dropout2 = nn.Dropout(p=0.3)

def forward(self, x):
    # Specify the forward pass, i.e. activation functions.
    x = torch.relu(self.fc1(x))
    x = self.dropout1(x)

In [ ]:
# Parameters (change and add as needed)
learning_rate = 0.01
num_epochs = 8000

In [ ]:
model = BinClassifierNN()

# choose a loss function for the classification
criterion =torch.nn.BCEWithLogitsLoss()


# choose an optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) #stochastic gradient descent optimizer:


> J did optimizer = optim.Adam(model.parameters(), learning_rate) 

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [18]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 1000 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.5f}')

Epoch [1000/10000], Loss: 0.37274
Epoch [2000/10000], Loss: 0.36288
Epoch [3000/10000], Loss: 0.34390
Epoch [4000/10000], Loss: 0.33174
Epoch [5000/10000], Loss: 0.31359
Epoch [6000/10000], Loss: 0.29178
Epoch [7000/10000], Loss: 0.27843
Epoch [8000/10000], Loss: 0.27085
Epoch [9000/10000], Loss: 0.24724
Epoch [10000/10000], Loss: 0.23133


> J v small loss bc did w/ batch training, =64

6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [31]:
from sklearn.metrics import accuracy_score, roc_auc_score
# Evaluate the model
bce_loss = torch.nn.BCEWithLogitsLoss()

model.eval()

with torch.no_grad():
    y_test_pred = model(X_test).squeeze()
    bce = bce_loss(y_test_pred, y_test).item()
    print("BCEWithLogitsLoss:", bce)


    y_pred_binary = (y_test_pred >= 0.5)
    test_acc = accuracy_score(y_test, y_pred_binary)

    #test_acc = accuracy_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_pred)


    #training
    y_train_pred = model(X_train).squeeze()
    y_train_binary = (y_train_pred >= 0.5)
    train_acc = accuracy_score(y_train, y_train_binary)

    #test_acc = accuracy_score(y_test, y_test_pred)
    train_auc = roc_auc_score(y_train, y_train_pred)

    bce_train = bce_loss(y_train_pred, y_train).item()
    
    # --- Test performance ---

    #y_test_prob = model(X_test).squeeze()[:, 1]
    
    #test_acc = accuracy_score(y_test, y_test_pred)
    #test_auc = roc_auc_score(y_test, y_test_prob)
    
    #print("BinaryNN")
    print("BCEWithLogitsLoss:", bce_train)
    print(f"  Test  Accuracy: {test_acc:.3f}")
    print(f"  Test  ROC-AUC:  {test_auc:.3f}")
    print(f"  Train  Accuracy: {train_acc:.3f}")

    print(f"  Train  ROC-AUC: {train_auc:.3f}")


    #print("-" * 40)

BCEWithLogitsLoss: 0.5979717969894409
BCEWithLogitsLoss: 0.23524689674377441
  Test  Accuracy: 0.771
  Test  ROC-AUC:  0.857
  Train  Accuracy: 0.880
  Train  ROC-AUC: 0.971


In [38]:
print(y_test)
print(y_test.sum())
print(len(y_test))
print(y_test_pred)
print(len(y_test)==len(y_test_pred))
print(bce)

print(y_test.type())
print(y_test_pred.type())


tensor([1., 1., 1.,  ..., 1., 1., 1.])
tensor(798.)
1456
tensor([0.1833, 0.1739, 0.1928,  ..., 0.1851, 0.1690, 0.1921])
True
0.6867174506187439
torch.FloatTensor
torch.FloatTensor


> ValueError: Classification metrics can't handle a mix of binary and continuous targets

# for all

In [32]:
from sklearn.metrics import accuracy_score, roc_auc_score

res = list(df.columns[4:])
y = df["mutagenicity"].to_numpy()

# Parameters (change and add as needed)
learning_rate = 0.01
num_epochs = 10000 



for fptype in res:
    #train test
    X = np.stack(df[fptype].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.

    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.float32)

    # Hyperparameters
    input_size = X_train.shape[1]
    hidden_size_1 = 35
    hidden_size_2 = 25
    output_size = 1

    

    # model stuff
    bce_loss = torch.nn.BCEWithLogitsLoss()
    model = BinClassifierNN()
    criterion =torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01) #stochastic gradient descent optimizer:

    #training optimiser
    for epoch in range(num_epochs):
        
        model.train()
        outputs = model(X_train).squeeze()
        loss = criterion(outputs, y_train)

        optimizer.zero_grad() # clear existing gradients
        loss.backward() # backpropagation of loss function to optimise gradient
        optimizer.step() # update parameters using Adam

        #if (epoch + 1) % 1000 == 0:
        #    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.5f}')

    #Evaluation time!!!
    model.eval()

    with torch.no_grad():
        y_test_pred = model(X_test).squeeze()
        bce = bce_loss(y_test_pred, y_test).item()

        y_pred_binary = (y_test_pred >= 0.5)
        test_acc = accuracy_score(y_test, y_pred_binary)
        test_auc = roc_auc_score(y_test, y_test_pred)

        #training
        y_train_pred = model(X_train).squeeze()
        bce_train = bce_loss(y_train_pred, y_train).item()
        y_train_binary = (y_train_pred >= 0.5)
        train_acc = accuracy_score(y_train, y_train_binary)
        train_auc = roc_auc_score(y_train, y_train_pred)


    print(f"{fptype}")
    print(f"  Train BCEWithLogitsLoss: {bce_train:.4f}")
    print(f"  Test BCEWithLogitsLoss: {bce:.4f}")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test  Accuracy: {test_acc:.4f}")
    print(f"  Train ROC-AUC:  {train_auc:.4f}")
    print(f"  Test  ROC-AUC:  {test_auc:.4f}")
    print("-" * 40)


MorganFP
  Train BCEWithLogitsLoss: 0.0866
  Test BCEWithLogitsLoss: 0.6064
  Train Accuracy: 0.9813
  Test  Accuracy: 0.8070
  Train ROC-AUC:  0.9961
  Test  ROC-AUC:  0.8763
----------------------------------------
MACCSkeys
  Train BCEWithLogitsLoss: 0.4310
  Test BCEWithLogitsLoss: 0.4777
  Train Accuracy: 0.8011
  Test  Accuracy: 0.7734
  Train ROC-AUC:  0.8827
  Test  ROC-AUC:  0.8523
----------------------------------------
RDKitFP
  Train BCEWithLogitsLoss: 0.1352
  Test BCEWithLogitsLoss: 0.5611
  Train Accuracy: 0.9555
  Test  Accuracy: 0.8180
  Train ROC-AUC:  0.9888
  Test  ROC-AUC:  0.8779
----------------------------------------
morganfp1024
  Train BCEWithLogitsLoss: 0.2310
  Test BCEWithLogitsLoss: 0.5172
  Train Accuracy: 0.9181
  Test  Accuracy: 0.7850
  Train ROC-AUC:  0.9644
  Test  ROC-AUC:  0.8594
----------------------------------------


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

When it comes to saving and loading models, there are three core functions to be familiar with:

torch.save: Saves a serialized object to disk. This function uses Python’s pickle utility for serialization. Models, tensors, and dictionaries of all kinds of objects can be saved using this function.
torch.load: Uses pickle’s unpickling facilities to deserialize pickled object files to memory. This function also facilitates the device to load the data into (see Saving & Loading Model Across Devices).
torch.nn.Module.load_state_dict: Loads a model’s parameter dictionary using a deserialized state_dict. For more information on state_dict, see What is a state_dict?.

In [41]:
# Print model's state_dict
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

# Print optimizer's state_dict
print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

Model's state_dict:
fc1.weight 	 torch.Size([25, 2048])
fc1.bias 	 torch.Size([25])
fc2.weight 	 torch.Size([15, 25])
fc2.bias 	 torch.Size([15])
fc3.weight 	 torch.Size([15, 15])
fc3.bias 	 torch.Size([15])
fco.weight 	 torch.Size([1, 15])
fco.bias 	 torch.Size([1])
Optimizer's state_dict:
state 	 {}
param_groups 	 [{'lr': 0.1, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1, 2, 3, 4, 5, 6, 7]}]


#### save

In [43]:

torch.save(model.state_dict(), "/Users/verityjanerothermelsmith/Documents/DSA104/DSA104/BinClassifierNN.pt")



#### load

In [44]:

model = BinClassifierNN()
model.load_state_dict(torch.load("/Users/verityjanerothermelsmith/Documents/DSA104/DSA104/BinClassifierNN.pt", weights_only=True))
model.eval()

BinClassifierNN(
  (fc1): Linear(in_features=2048, out_features=25, bias=True)
  (fc2): Linear(in_features=25, out_features=15, bias=True)
  (fc3): Linear(in_features=15, out_features=15, bias=True)
  (fco): Linear(in_features=15, out_features=1, bias=True)
)

#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


- MorganFP \
  Train BCEWithLogitsLoss: 0.0866 \
  Test BCEWithLogitsLoss: 0.6064 \
  Train Accuracy: 0.9813 \
  Test  Accuracy: 0.8070 \
  Train ROC-AUC:  0.9961\
  Test  ROC-AUC:  0.8763\

- MACCSkeys\
  Train BCEWithLogitsLoss: 0.4310\
  Test BCEWithLogitsLoss: 0.4777\
  Train Accuracy: 0.8011\
  Test  Accuracy: 0.7734\
  Train ROC-AUC:  0.8827\
  Test  ROC-AUC:  0.8523\

- RDKitFP\
  Train BCEWithLogitsLoss: 0.1352\
  Test BCEWithLogitsLoss: 0.5611\
  Train Accuracy: 0.9555\
  Test  Accuracy: 0.8180\
  Train ROC-AUC:  0.9888\
  Test  ROC-AUC:  0.8779\

- morganfp1024\
  Train BCEWithLogitsLoss: 0.2310\
  Test BCEWithLogitsLoss: 0.5172\
  Train Accuracy: 0.9181\
  Test  Accuracy: 0.7850\
  Train ROC-AUC:  0.9644\
  Test  ROC-AUC:  0.8594


1) I'm not sure I calc. Accuracy and ROC-AUC properly, the model performs okay, but considering the effort and time it takes to calculate it is just okay. The test accuary was 0.8 and ROC-AUC was .88. Therefore it underperformed in regard to the random forest model, but outperformed the others. 
2) Yes, the RDKit had slightly higher test metric values,(0.8180, 0.8779) albeit a higher BCE value that the MACCSkexs (0.5611 - 0.4777)

3) No, not really. The 1024 bit data performed the same as 2028 bits, Very comparable. Lesson maybe that we can save on computing by using 1024?

4) lr = 0.01, I decreased the hidden layer size on the thirs layer from 35 to 25. Epocs 10000 very good. Biggest effect was increasing hdden layers from 15/25 to 25/35. I didn't experiment with the optimiser.

5) Yes, overfitting was present on Morgan, RDKit and morgan1024. MACCS has about 0.3% overfitting which is acceptable for me. Maybe it could be limited by decreasing certain parameters so it can't over correct as much? Could test out with diff lr etc. 

6) Noise measn structures that don't have an influence on mutagenicity. Yes and no use in lab, I guess if I was doing this research it would give me preliminary values. They wouldn't be suffieienct for pharma though, as the mutagenicity is usually confirmed and calced using experimental data (?). 
Data driven tools for QSAR: binding constants, PSA, LogP etc. and of course binding pockets and interaction simulation etc. 

7) Memory, if we just save the params then "When the model is loaded, a new model with the same architecture is created. Then, the parameters of the new model are replaced with the stored parameters. Since only parameters are stored, this method is memory efficient. The following code snippet illustrates this method."

Side note, I've n ot saved the models I made here, could and shoulld be implemented in the loop. Some extenal fct would also be good, esp. for visualation of accuacy. Prediction matrix would be coool.

<blockquote> more layers -> more overfitting

dropout: deactivate a certain % of neurons per layer.

Why do? no imprint certain patterns that it can stick too, less overfitting

6) danger: false positives, measurement errors, nat. distributions, ground truth may still be hidden

also clustering etc

